In [ ]:
# load the complete dataset from csv file
# generate isomap and UMAP values of the sub-groups
# save results as isomap and UMAP csv files for Moving Average generation

# select different sub-groups (eg. ctl and SCA1)
# combine with the corresponding dataset INFO
# write out the subject names as csv if needed (eg. ctl_sca1_time12)

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

In [2]:
curProject = 'ataxia'
curRoot = 'C'  # 'C' or 'D'

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def perform_pca(dist_df, numDim):
    """
    Performs PCA on a dataframe of descriptors. 
    Always uses Euclidean distance by its nature, highly stable for small samples.
    """
    subjNames = dist_df.index
    dimNames = np.arange(1, numDim + 1)
    columns = [f"pca_dim{i}" for i in dimNames]
    
    # Standardize the data (Crucial for PCA), each of the descriptors has a mean of 0 and variance of 1
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(dist_df.values)
    
    # Initialize and Run PCA
    pca_model = PCA(n_components=numDim)
    pca_values = pca_model.fit_transform(X_scaled)
    
    # Create DataFrame
    pca_DF = pd.DataFrame(pca_values, index=subjNames, columns=columns)
    
    # Attach metadata: Explained Variance Ratio, how much "information" is captured in the numDim dimensions
    explained_var = pca_model.explained_variance_ratio_.sum()
    pca_DF.attrs['explained_variance'] = explained_var
    
    print(f"Total Explained Variance (3 dims): {explained_var:.2%}")
    
    return pca_DF

In [122]:
################################################    generation of Isomap    ###############################################
from sklearn.manifold import Isomap

def perform_isomap(dist,numDim,numNeig,metric='euclidean'):
    subjNames = dist.index
    dimNames = np.arange(1,numDim+1)
    columns = [f"iso_dim{i}_neig{numNeig}" for i in dimNames]
    #print(columns)    

    dist_centered = dist.values - dist.values.mean(axis=0)
    #iso = Isomap(n_neighbors=numNeig,n_components=numDim).fit_transform(dist.values)
    iso = Isomap(n_neighbors=numNeig,n_components=numDim,metric=metric).fit_transform(dist_centered)
    iso_DF = pd.DataFrame(iso,index=subjNames,columns=columns)    
    return iso_DF
    
def perform_region_isomap(dist,numDim_iso,numNeig_list_iso,curRegion,writeIsomap,metric='euclidean'):
    iso = pd.DataFrame(index=dist.index)
    print('Generating isomaps.')
    for numNeig in numNeig_list_iso:
        iso_cur = perform_isomap(dist,numDim_iso,numNeig,metric=metric)
        # add to existing df
        iso_DF = pd.DataFrame(iso_cur, index=iso.index)
        iso = pd.concat([iso, iso_DF], axis=1)
    if writeIsomap:  # SAVE Isomaps as csv, for debug
        outName = 'iso_'+curRegion+'_dim_'+str(numDim_iso)+'_neig_'+str(numNeig_list_iso)+'.csv'
        outFileName = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Atril_Biosca_Cermoi_iso_u\{outName}'
        print(outFileName)
        iso.to_csv(outFileName,index_label='subjID')
    return iso

In [77]:
###############################   generation of UMAP   ################################
import umap
import random
# to ensure that the results are always the same
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

def perform_UMAP(df,n_comp,n_neighbors,min_dist):
    reducer = umap.UMAP(n_components=n_comp,n_neighbors=n_neighbors,min_dist=min_dist,random_state=SEED)
    embedding = reducer.fit_transform(df)

    columns = []    
    if n_comp == 1:    # Column naming logic
        columns = [f'u_dim{n_comp}_neig{n_neighbors}']
    elif n_comp == 2:
        columns = [f'u_dim{n_comp}_1_neig{n_neighbors}', f'u_dim{n_comp}_2_neig{n_neighbors}']
    else:
        columns = [f'u_dim{n_comp}_{i}_neig{n_neighbors}' for i in range(n_comp)]
    
    embedding_df = pd.DataFrame(embedding, columns=columns)   # Create a DataFrame for the embedding
    embedding_df.index = df.index
    return embedding_df

def perform_region_UMAP(dist,curRegion,numDim_u,numNeig_list_u,writeUMAP):
    umap_results = pd.DataFrame(index=dist.index)
    min_dist = 0.2                # Change this to the desired minimum distance
    print('Generating UMAPs.')
    for n_comp in numDim_u:
        for n_neighbors in numNeig_list_u:  # perform UMAP
            embedding_df = perform_UMAP(dist, n_comp, n_neighbors, min_dist)   # Call the helper function                     
            umap_results = pd.concat([umap_results, embedding_df], axis=1)    # Concatenate to our results dataframe
    if writeUMAP:  # SAVE Umap as csv, for debug
        outName = 'u_'+curRegion+'_dim_'+str(numDim_u)+'_neig_'+str(numNeig_list_u)+'.csv'
        outFileName = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Atril_Biosca_Cermoi_iso_u\{outName}'
        print(outFileName)
        umap_results.to_csv(outFileName,index_label='subjID')
    return umap_results

In [8]:
# read in region names #
region_file = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\region_names_Atril.csv'
regions = None
try:
    regions = pd.read_csv(region_file, index_col=0,header=0)
except FileNotFoundError as e:
    print(f"Error: {e}")

#print(regions)

regions_Atril = ["SC-sylv_right_name06-17-02--84_embeddings", "SFint-SR_right_name08-12-43--252_embeddings", 
                 "SCall_left_name19-33-54--73_embeddings", "OCCIPITAL_left_name07-34-40--229_embeddings", 
                 "SPeC_left_name08-24-23--109_embeddings", "SFmedian-SFpoltr-SFsup_right_name08-19-50--180_embeddings", 
                 "SOr-SOlf_left_name08-22-26--235_embeddings", "new_regions_SCall--right--name09-24-24--37_embeddings", 
                 "new_regions_FIP-FIPPoCinf--right--name13-52-34--184_embeddings", "STi-STs-STpol_right_name16-06-03--189_embeddings", 
                 "SC-SPeC_right_name06-17-00--24_embeddings", "SPoC_right_name08-28-01--155_embeddings", 
                 "FPO-SCu-ScCal_right_name07-15-26--174_embeddings", "SCall_right_name09-24-24--37_embeddings", 
                 "SFinf-BROCA-SPeCinf_right_name08-00-44--234_embeddings", "FColl-SRh_right_name06-56-15--113_embeddings", 
                 "SFinter-SFsup_right_name08-08-42--126_embeddings", "SFmedian-SFpoltr-SFsup_left_name06-17-02--114_embeddings", 
                 "OCCIPITAL_right_name07-38-28--182_embeddings", "ScCal-SLi_left_name07-41-43--180_embeddings", 
                 "SC-sylv_left_name07-58-00--111_embeddings", "SFint-FCMant_left_name08-09-20--81_embeddings", 
                 "SCall-SsP-SintraCing_right_name18-49-13--26_embeddings", "SFint-SR_left_name08-11-23--65_embeddings", 
                 "Lobule_parietal_sup_right_name07-24-01--193_embeddings", "STsbr_left_name08-31-32--170_embeddings", 
                 "FIP-FIPPoCinf_left_name13-52-34--154_embeddings", "ScCal-SLi_right_name07-54-42--78_embeddings", 
                 "STsbr_right_name08-32-57--200_embeddings", "Lobule_parietal_sup_left_name07-23-04--36_embeddings", 
                 "STi-SOTlat_right_name06-17-38--74_embeddings", "new_regions_SCall-SsP-SintraCing--left--name10-52-10--210_embeddings",
                 "SOr-SOlf_right_name08-22-27--80_embeddings", "STs_right_name08-32-58--52_embeddings", 
                 "SFint-FCMant_right_name08-10-30--253_embeddings", "SC-SPoC_left_name07-57-18--53_embeddings", 
                 "SFinf-BROCA-SPeCinf_left_name08-00-45--128_embeddings", "FCMpost-SpC_left_name06-21-10--231_embeddings", 
                 "STi-STs-STpol_left_name06-17-41--148_embeddings", "SFmarginal-SFinfant_left_name08-15-17--25_embeddings", 
                 "SPoC_left_name08-26-18--11_embeddings", "SsP-SPaint_left_name08-28-13--26_embeddings", 
                 "SC-SPoC_right_name07-58-03--243_embeddings", "FIP-FIPPoCinf_right_name13-5F2-34--184_embeddings", 
                 "SFinter-SFsup_left_name08-06-01--220_embeddings", "SsP-SPaint_right_name08-29-38--71_embeddings", 
                 "SC-SPeC_left_name22-16-47--177_embeddings", "new_regions_FIP-FIPPoCinf--left--name13-52-34--154_embeddings", 
                 "FCMpost-SpC_right_name06-34-24--229_embeddings", "FCLp-subsc-FCLa-INSULA_right_name17-47-16--166_embeddings", 
                 "new_regions_SCall--left--name19-33-54--73_embeddings", "SOr_right_name14-12-56--58_embeddings", 
                 "new_regions_SCall-SsP-SintraCing--right--name18-49-13--26_embeddings", 
                 "SFmarginal-SFinfant_right_name08-17-15--135_embeddings", "FColl-SRh_left_name06-43-43--210_embeddings", 
                 "SOr_left_name14-12-56--162_embeddings", "SPeC_right_name08-24-57--227_embeddings", 
                 "STs_left_name08-32-58--223_embeddings", "FCLp-subsc-FCLa-INSULA_left_name17-43-58--232_embeddings", 
                 "SCall-SsP-SintraCing_left_name10-52-10--210_embeddings", "FPO-SCu-ScCal_left_name07-13-21--118_embeddings", 
                 "STi-SOTlat_left_name06-17-04--76_embeddings"]
regions_Biosca = ["SC-sylv_right_name06-17-02--84_embeddings", "SFint-SR_right_name08-12-43--252_embeddings", "SCall_left_name19-33-54--73_embeddings", "OCCIPITAL_left_name07-34-40--229_embeddings", "SPeC_left_name08-24-23--109_embeddings", "SFmedian-SFpoltr-SFsup_right_name08-19-50--180_embeddings", "SOr-SOlf_left_name08-22-26--235_embeddings", "new_regions_SCall--right--name09-24-24--37_embeddings", "new_regions_FIP-FIPPoCinf--right--name13-52-34--184_embeddings", "STi-STs-STpol_right_name16-06-03--189_embeddings", "SC-SPeC_right_name06-17-00--24_embeddings", "SPoC_right_name08-28-01--155_embeddings", "FPO-SCu-ScCal_right_name07-15-26--174_embeddings", "SCall_right_name09-24-24--37_embeddings", "SFinf-BROCA-SPeCinf_right_name08-00-44--234_embeddings", "FColl-SRh_right_name06-56-15--113_embeddings", "SFinter-SFsup_right_name08-08-42--126_embeddings", "SFmedian-SFpoltr-SFsup_left_name06-17-02--114_embeddings", "OCCIPITAL_right_name07-38-28--182_embeddings", "ScCal-SLi_left_name07-41-43--180_embeddings", "SC-sylv_left_name07-58-00--111_embeddings", "SFint-FCMant_left_name08-09-20--81_embeddings", "SCall-SsP-SintraCing_right_name18-49-13--26_embeddings", "SFint-SR_left_name08-11-23--65_embeddings", "Lobule_parietal_sup_right_name07-24-01--193_embeddings", "STsbr_left_name08-31-32--170_embeddings", "FIP-FIPPoCinf_left_name13-52-34--154_embeddings", "ScCal-SLi_right_name07-54-42--78_embeddings", "STsbr_right_name08-32-57--200_embeddings", "Lobule_parietal_sup_left_name07-23-04--36_embeddings", "STi-SOTlat_right_name06-17-38--74_embeddings", "new_regions_SCall-SsP-SintraCing--left--name10-52-10--210_embeddings", "SOr-SOlf_right_name08-22-27--80_embeddings", "STs_right_name08-32-58--52_embeddings", "SFint-FCMant_right_name08-10-30--253_embeddings", "SC-SPoC_left_name07-57-18--53_embeddings", "SFinf-BROCA-SPeCinf_left_name08-00-45--128_embeddings", "FCMpost-SpC_left_name06-21-10--231_embeddings", "STi-STs-STpol_left_name06-17-41--148_embeddings", "SFmarginal-SFinfant_left_name08-15-17--25_embeddings", "SPoC_left_name08-26-18--11_embeddings", "SsP-SPaint_left_name08-28-13--26_embeddings", "SC-SPoC_right_name07-58-03--243_embeddings", "FIP-FIPPoCinf_right_name13-52-34--184_embeddings", "SFinter-SFsup_left_name08-06-01--220_embeddings", "SsP-SPaint_right_name08-29-38--71_embeddings", "SC-SPeC_left_name22-16-47--177_embeddings", "new_regions_FIP-FIPPoCinf--left--name13-52-34--154_embeddings", "FCMpost-SpC_right_name06-34-24--229_embeddings", "FCLp-subsc-FCLa-INSULA_right_name17-47-16--166_embeddings", "new_regions_SCall--left--name19-33-54--73_embeddings", "SOr_right_name14-12-56--58_embeddings", "new_regions_SCall-SsP-SintraCing--right--name18-49-13--26_embeddings", "SFmarginal-SFinfant_right_name08-17-15--135_embeddings", "FColl-SRh_left_name06-43-43--210_embeddings", "SOr_left_name14-12-56--162_embeddings", "SPeC_right_name08-24-57--227_embeddings", "STs_left_name08-32-58--223_embeddings", "FCLp-subsc-FCLa-INSULA_left_name17-43-58--232_embeddings", "SCall-SsP-SintraCing_left_name10-52-10--210_embeddings", "FPO-SCu-ScCal_left_name07-13-21--118_embeddings", "STi-SOTlat_left_name06-17-04--76_embeddings"]
regions_Cermoi = ["SC-sylv_right_name06-17-02--84_embeddings", "SFint-SR_right_name08-12-43--252_embeddings", "SCall_left_name19-33-54--73_embeddings", "OCCIPITAL_left_name07-34-40--229_embeddings", "SPeC_left_name08-24-23--109_embeddings", "SFmedian-SFpoltr-SFsup_right_name08-19-50--180_embeddings", "SOr-SOlf_left_name08-22-26--235_embeddings", "new_regions_SCall--right--name09-24-24--37_embeddings", "new_regions_FIP-FIPPoCinf--right--name13-52-34--184_embeddings", "STi-STs-STpol_right_name16-06-03--189_embeddings", "SC-SPeC_right_name06-17-00--24_embeddings", "SPoC_right_name08-28-01--155_embeddings", "FPO-SCu-ScCal_right_name07-15-26--174_embeddings", "SCall_right_name09-24-24--37_embeddings", "SFinf-BROCA-SPeCinf_right_name08-00-44--234_embeddings", "FColl-SRh_right_name06-56-15--113_embeddings", "SFinter-SFsup_right_name08-08-42--126_embeddings", "SFmedian-SFpoltr-SFsup_left_name06-17-02--114_embeddings", "OCCIPITAL_right_name07-38-28--182_embeddings", "ScCal-SLi_left_name07-41-43--180_embeddings", "SC-sylv_left_name07-58-00--111_embeddings", "SFint-FCMant_left_name08-09-20--81_embeddings", "SCall-SsP-SintraCing_right_name18-49-13--26_embeddings", "SFint-SR_left_name08-11-23--65_embeddings", "Lobule_parietal_sup_right_name07-24-01--193_embeddings", "STsbr_left_name08-31-32--170_embeddings", "FIP-FIPPoCinf_left_name13-52-34--154_embeddings", "ScCal-SLi_right_name07-54-42--78_embeddings", "STsbr_right_name08-32-57--200_embeddings", "Lobule_parietal_sup_left_name07-23-04--36_embeddings", "STi-SOTlat_right_name06-17-38--74_embeddings", "new_regions_SCall-SsP-SintraCing--left--name10-52-10--210_embeddings", "SOr-SOlf_right_name08-22-27--80_embeddings", "STs_right_name08-32-58--52_embeddings", "SFint-FCMant_right_name08-10-30--253_embeddings", "SC-SPoC_left_name07-57-18--53_embeddings", "SFinf-BROCA-SPeCinf_left_name08-00-45--128_embeddings", "FCMpost-SpC_left_name06-21-10--231_embeddings", "STi-STs-STpol_left_name06-17-41--148_embeddings", "SFmarginal-SFinfant_left_name08-15-17--25_embeddings", "SPoC_left_name08-26-18--11_embeddings", "SsP-SPaint_left_name08-28-13--26_embeddings", "SC-SPoC_right_name07-58-03--243_embeddings", "FIP-FIPPoCinf_right_name13-52-34--184_embeddings", "SFinter-SFsup_left_name08-06-01--220_embeddings", "SsP-SPaint_right_name08-29-38--71_embeddings", "SC-SPeC_left_name22-16-47--177_embeddings", "new_regions_FIP-FIPPoCinf--left--name13-52-34--154_embeddings", "FCMpost-SpC_right_name06-34-24--229_embeddings", "FCLp-subsc-FCLa-INSULA_right_name17-47-16--166_embeddings", "new_regions_SCall--left--name19-33-54--73_embeddings", "SOr_right_name14-12-56--58_embeddings", "new_regions_SCall-SsP-SintraCing--right--name18-49-13--26_embeddings", "SFmarginal-SFinfant_right_name08-17-15--135_embeddings", "FColl-SRh_left_name06-43-43--210_embeddings", "SOr_left_name14-12-56--162_embeddings", "SPeC_right_name08-24-57--227_embeddings", "STs_left_name08-32-58--223_embeddings", "FCLp-subsc-FCLa-INSULA_left_name17-43-58--232_embeddings", "SCall-SsP-SintraCing_left_name10-52-10--210_embeddings", "FPO-SCu-ScCal_left_name07-13-21--118_embeddings", "STi-SOTlat_left_name06-17-04--76_embeddings"]
regions = regions_Atril
left  = [x for x in regions if "left" in x.lower()]
right = [x for x in regions if "right" in x.lower()]
other = [x for x in regions if "left" not in x.lower() and "right" not in x.lower()]

print("LEFT:", len(left))
print("RIGHT:", len(right))
print("OTHER:", len(other))

def strip_after_side(name):
    low = name.lower()
    if "left" in low:
        return name[:low.index("left")].rstrip("_-")
    elif "right" in low:
        return name[:low.index("right")].rstrip("_-")
    else:
        return name

region_without_leftRight = [strip_after_side(x) for x in left]
print(region_without_leftRight)
print(len(region_without_leftRight))


LEFT: 31
RIGHT: 31
OTHER: 0
['SCall', 'OCCIPITAL', 'SPeC', 'SOr-SOlf', 'SFmedian-SFpoltr-SFsup', 'ScCal-SLi', 'SC-sylv', 'SFint-FCMant', 'SFint-SR', 'STsbr', 'FIP-FIPPoCinf', 'Lobule_parietal_sup', 'new_regions_SCall-SsP-SintraCing', 'SC-SPoC', 'SFinf-BROCA-SPeCinf', 'FCMpost-SpC', 'STi-STs-STpol', 'SFmarginal-SFinfant', 'SPoC', 'SsP-SPaint', 'SFinter-SFsup', 'SC-SPeC', 'new_regions_FIP-FIPPoCinf', 'new_regions_SCall', 'FColl-SRh', 'SOr', 'STs', 'FCLp-subsc-FCLa-INSULA', 'SCall-SsP-SintraCing', 'FPO-SCu-ScCal', 'STi-SOTlat']
31


In [311]:
# Optional: save the region names without left/right

print(type(region_without_leftRight))
print(region_without_leftRight)

#region_without_leftRight = pd.DataFrame(region_without_leftRight, columns=['regionRoot'])
region_without_leftRight = pd.DataFrame(region_without_leftRight)
outFileName = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\region_names_withoutLeftRight.csv'
print(outFileName)
#region_without_leftRight.to_csv(outFileName)

<class 'pandas.core.frame.DataFrame'>
                          regionRoot
0                              SCall
1                          OCCIPITAL
2                               SPeC
3                           SOr-SOlf
4             SFmedian-SFpoltr-SFsup
5                          ScCal-SLi
6                            SC-sylv
7                       SFint-FCMant
8                           SFint-SR
9                              STsbr
10                     FIP-FIPPoCinf
11               Lobule_parietal_sup
12  new_regions_SCall-SsP-SintraCing
13                           SC-SPoC
14               SFinf-BROCA-SPeCinf
15                       FCMpost-SpC
16                     STi-STs-STpol
17               SFmarginal-SFinfant
18                              SPoC
19                        SsP-SPaint
20                     SFinter-SFsup
21                           SC-SPeC
22         new_regions_FIP-FIPPoCinf
23                 new_regions_SCall
24                         FColl-SRh


In [360]:
#######################   create iso_u directory to write output if needed   ########################
outFileDir = rf"{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Atril_Biosca_Cermoi_iso_u"
print(outFileDir)
os.makedirs(outFileDir, exist_ok=True)

# create iso_u_withDBinfo directory to write output if needed
outFileDir = rf"{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Atril_Biosca_Cermoi_iso_u_with_DB_info"
print(outFileDir)
os.makedirs(outFileDir, exist_ok=True)


C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_iso_u
C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_iso_u_with_DBinfo


In [87]:
#######################   create iso_u directory to write output of specific SCAs if needed   ########################
sca_numbers = [1, 2, 3, 7]

for sca in sca_numbers:
    # Construct the path dynamically
    path = rf"{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Atril_Biosca_Cermoi_iso_u_sca_{sca}"
    path_with_DB = rf"{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Atril_Biosca_Cermoi_iso_u_with_info_sca_{sca}"
    
    print(f"Creating directory for SCA {sca}: {path}")
    os.makedirs(path, exist_ok=True)
    print(f"Creating directory for SCA {sca}: {path_with_DB}")
    os.makedirs(path_with_DB, exist_ok=True)

Creating directory for SCA 1: C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_iso_u_sca_1
Creating directory for SCA 1: C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_iso_u_with_info_sca_1
Creating directory for SCA 2: C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_iso_u_sca_2
Creating directory for SCA 2: C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_iso_u_with_info_sca_2
Creating directory for SCA 3: C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_iso_u_sca_3
Creating directory for SCA 3: C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_iso_u_with_info_sca_3
Creating directory for SCA 7: C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_iso_u_sca_7
Creating directory for SCA 7: C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_iso_u_with_info_sca_7


In [13]:
#######################   read in all_DB_info csv   ########################
infoBaselineName = rf'{curRoot}:\B_projWIP\proj_{curProject}\SCA_INFO\processed_INFO\ATRIL_BIOSCA_CERMOI_time1_only.csv'
Atril_Biosca_Cermoi_INFO = pd.read_csv(infoBaselineName,index_col=0,header=0)

#print("combined_iso_u columns:", combined_iso_u.columns.tolist())
print("INFO columns:", Atril_Biosca_Cermoi_INFO.columns.tolist())

### verification of column names ###
#one_iso_u_name = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Atril_Biosca_Cermoi_iso_u\FColl-SRh_left_name06-43-43--210_embeddings_iso_u.csv'
#one_combined_iso_u = pd.read_csv(one_iso_u_name,index_col=0,header=0)
#print("region columns:", one_combined_iso_u.columns.tolist())

INFO columns: ['RANDOMIZATION', 'SCA', 'CAG', 'Sex', 'Age', 'SARA', 'INAS', 'CodeICM', 'Age_onset', 'CCFS', 'Handedness', 'Disease_duration', 'allele1', 'allele2', 'minAllele', 'maxAllele', 'sumAllele']


In [97]:
###########################################################################################################
## Note that this is the version with all SCAs together, separating SCA types refer to code that follows ##
## for each region, reading distance files of Atril, Biosca, Cermoi, combine to one distance file        ##
##                  perform isomap and umap                                                              ##
##                  combine this iso_u                                                                   ##
###########################################################################################################

numDim_iso = 3
numNeig_list_iso = {10,30}     # MODIFY if needed
numDim_u = {1,2}
numNeig_list_u = {10,30}       # MODIFY if needed

for curRegion in regions:
    dist_Atril, dist_Biosca, dist_Cermoi = None, None, None    
    distance_path_Atril = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Atril\{curRegion}.csv'
    distance_path_Biosca = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Biosca\{curRegion}.csv'
    distance_path_Cermoi = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Cermoi\{curRegion}.csv'
    try:
        dist_Atril = pd.read_csv(distance_path_Atril, index_col=0, header=0)
        dist_Biosca = pd.read_csv(distance_path_Biosca, index_col=0, header=0)
        dist_Cermoi = pd.read_csv(distance_path_Cermoi, index_col=0, header=0)
    except FileNotFoundError as e:
        print(f"Error: {e}")
        
    dist_Atril_Biosca_Cermoi = pd.concat([dist_Atril, dist_Biosca, dist_Cermoi], axis=0)

    iso = perform_region_isomap(dist_Atril_Biosca_Cermoi,numDim_iso,numNeig_list_iso,curRegion,writeIsomap=False)  # write true to debug iso file
    u = perform_region_UMAP(dist_Atril_Biosca_Cermoi,curRegion,numDim_u,numNeig_list_u,writeUMAP=False)            # write true to debug umap file
    cur_combined_iso_u = pd.concat([iso,u], axis=1)

    # write out the iso_u file for one region
    combined_iso_u_fileName = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Atril_Biosca_Cermoi_iso_u\{curRegion}_iso_u.csv'
#    print(combined_iso_u_fileName)
    cur_combined_iso_u.to_csv(combined_iso_u_fileName,index_label='subjID')    

    ###  merging with DB INFO  ###
    combined_iso_u_INFO = pd.merge(
    cur_combined_iso_u,
    Atril_Biosca_Cermoi_INFO,
    left_index=True,  #  subjID is the index name of left df
    right_index=True  # subjID is the index name of the right df
    )
    # write out the iso_u file with INFO for one region
    combined_iso_u_INFO_fileName = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Atril_Biosca_Cermoi_iso_u_with_DB_INFO\{curRegion}_iso_u_INFO.csv'
    print(combined_iso_u_INFO_fileName)
    combined_iso_u_INFO.to_csv(combined_iso_u_INFO_fileName,index_label='subjID')

Generating isomaps.
Generating UMAPs.


C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_iso_u_with_DB_INFO\SC-sylv_right_name06-17-02--84_embeddings_iso_u_INFO.csv
Generating isomaps.
Generating UMAPs.


C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_iso_u_with_DB_INFO\SFint-SR_right_name08-12-43--252_embeddings_iso_u_INFO.csv
Generating isomaps.
Generating UMAPs.


...

C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_iso_u_with_DB_INFO\SCall-SsP-SintraCing_left_name10-52-10--210_embeddings_iso_u_INFO.csv
Generating isomaps.
Generating UMAPs.


C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_iso_u_with_DB_INFO\FPO-SCu-ScCal_left_name07-13-21--118_embeddings_iso_u_INFO.csv
Generating isomaps.
Generating UMAPs.


C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_iso_u_with_DB_INFO\STi-SOTlat_left_name06-17-04--76_embeddings_iso_u_INFO.csv


In [101]:
##########################  Define subject lists for distance selection  #############################

# 1. Get the relevant subjects
sca_1_ctl = (Atril_Biosca_Cermoi_INFO['SCA'] == 1) | \
            ((Atril_Biosca_Cermoi_INFO['SCA'] == 0) & (Atril_Biosca_Cermoi_INFO['CodeICM'] == 'BIOSCA'))
sca_2_ctl = ((Atril_Biosca_Cermoi_INFO['SCA'] == 2) & (Atril_Biosca_Cermoi_INFO['CodeICM'] != 'ATRIL')) | \
            (Atril_Biosca_Cermoi_INFO['SCA'] == 0) 
sca_2_ctl_with_Atril = (Atril_Biosca_Cermoi_INFO['SCA'] == 2) | (Atril_Biosca_Cermoi_INFO['SCA'] == 0) 
sca_3_ctl = (Atril_Biosca_Cermoi_INFO['SCA'] == 3) | \
            ((Atril_Biosca_Cermoi_INFO['SCA'] == 0) & (Atril_Biosca_Cermoi_INFO['CodeICM'] == 'BIOSCA'))
sca_7_ctl = (Atril_Biosca_Cermoi_INFO['SCA'] == 7) | (Atril_Biosca_Cermoi_INFO['SCA'] == 0) 

# 2. Use the condition to get the index (subject names)
subjects_sca_1_ctl = Atril_Biosca_Cermoi_INFO.loc[sca_1_ctl].index.tolist()
#print(subjects_sca_1_ctl)
subjects_sca_2_ctl = Atril_Biosca_Cermoi_INFO.loc[sca_2_ctl].index.tolist()
print(subjects_sca_2_ctl)
subjects_sca_2_ctl_with_Atril = Atril_Biosca_Cermoi_INFO.loc[sca_2_ctl_with_Atril].index.tolist()
print(subjects_sca_2_ctl_with_Atril
     )
subjects_sca_3_ctl = Atril_Biosca_Cermoi_INFO.loc[sca_3_ctl].index.tolist()
#print(subjects_sca_3_ctl)
subjects_sca_7_ctl = Atril_Biosca_Cermoi_INFO.loc[sca_7_ctl].index.tolist()
#print(subjects_sca_7_ctl)


['001012FJ', '001015VJ', '001017VP', '001019DA', '001020HG', '001021CJ', '001022LM', '001025LJ', '001027RY', '001032SG', '001037GA', '001040BF', '001045PB', '001046CJ', '001049BD', '001054MP', '001055JC', '001057MB', '001058FG', '001059MV', '001060MJ', '001065BC', '001073PM', '001075HJ', '001078PM', '001079LP', '001085BN', '001086CP', '001091MR', '001099GL', '001100PY', '001101JO', '00001PJ', '00002PV', '00004PA', '00007OP', '00008CJ', '00009LN', '00011EG', '00012BM', '00017ML', '00019RP', '00020CT', '00021JA', '00022DA', '00023EA', '00025AY', '00026AD', '00027EF', '00029DP', '00030CA', '00031CP', '00032DL', '00035NR', '00036DC', '00037CI', '00039OV']
['0010001OP', '0010002MV', '0010003CJ', '0010004HV', '0010005BC', '0010006OG', '0010007MA', '0010008CT', '0010009BJ', '0010010DM', '0010011CP', '0010012MC', '0010013AN', '0010014MM', '0010015BV', '0010016VP', '0010017MD', '0010018RE', '0010019MM', '0010020PA', '0010021MA', '0010022MB', '0010023MM', '0010024VN', '0010025RA', '0010026DA', '

In [83]:
#######################  Distance selection test  #########################
test_region = 'SC-sylv_right_name06-17-02--84_embeddings'
distance_path_Atril = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Atril\{test_region}.csv'
distance_path_Biosca = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Biosca\{test_region}.csv'
distance_path_Cermoi = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Cermoi\{test_region}.csv'
try:
    dist_Atril = pd.read_csv(distance_path_Atril, index_col=0, header=0)
    dist_Biosca = pd.read_csv(distance_path_Biosca, index_col=0, header=0)
    dist_Cermoi = pd.read_csv(distance_path_Cermoi, index_col=0, header=0)
except FileNotFoundError as e:
    print(f"Error: {e}")
dist_Atril_Biosca_Cermoi = pd.concat([dist_Atril, dist_Biosca, dist_Cermoi], axis=0)

valid_subjects = dist_Atril_Biosca_Cermoi.index.intersection(subjects_sca_1_ctl)
dist_sca1 = dist_Atril_Biosca_Cermoi.loc[valid_subjects,]
print(len(dist_sca1))

34


In [126]:
####################################################################################################
##                           same as above, but for specific SCAs                                 ##
## for each region, reading distance files of Atril, Biosca, Cermoi, combine to one distance file ##
##                  perform isomap and umap                                                       ##
##                  combine this iso_u                                                            ##
####################################################################################################
sca_targets = [1, 2, 3, 7]
#sca_targets = [102] # 102: sca 2 Biosca + Control Biosca + Atril

sca_dir_map = {}

metric = 'manhattan' # isomap metrics: 'cosine', 'manhattan', 'euclidean'
numDim_iso = 3
numNeig_list_iso = {5,30}     # MODIFY if needed
numDim_u = {1,2}
numNeig_list_u = {10,30}       # MODIFY if needed

for sca in sca_targets:
    path = rf"{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Atril_Biosca_Cermoi_iso_u_sca_{sca}"
    sca_dir_map[sca] = path

    for curRegion in regions:
        combined_iso_u = None
        dist_Atril, dist_Biosca, dist_Cermoi = None, None, None    
        distance_path_Atril = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Atril\{curRegion}.csv'
        distance_path_Biosca = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Biosca\{curRegion}.csv'
        distance_path_Cermoi = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Cermoi\{curRegion}.csv'
        try:
            dist_Atril = pd.read_csv(distance_path_Atril, index_col=0, header=0)
            dist_Biosca = pd.read_csv(distance_path_Biosca, index_col=0, header=0)
            dist_Cermoi = pd.read_csv(distance_path_Cermoi, index_col=0, header=0)
        except FileNotFoundError as e:
            print(f"Error: {e}")
        dist_Atril_Biosca_Cermoi = pd.concat([dist_Atril, dist_Biosca, dist_Cermoi], axis=0)
        #########################  Selection of subjects  ########################
        subjects_select = []
        if sca == 1:
            subjects_select = subjects_sca_1_ctl
        if sca == 2:
            subjects_select = subjects_sca_2_ctl
        if sca == 3:
            subjects_select = subjects_sca_3_ctl
        if sca == 7:
            subjects_select = subjects_sca_7_ctl  
        if sca == 102:
            subjects_select = subjects_sca_2_ctl_with_Atril              
        valid_subjects = dist_Atril_Biosca_Cermoi.index.intersection(subjects_select)
        dist_select = dist_Atril_Biosca_Cermoi.loc[valid_subjects,]

        # Calculate isomap, umap
        iso = perform_region_isomap(dist_select,numDim_iso,numNeig_list_iso,curRegion,writeIsomap=False,metric=metric) # write true to debug iso file
        u = perform_region_UMAP(dist_select,curRegion,numDim_u,numNeig_list_u,writeUMAP=False)           # write true to debug umap file
        cur_combined_iso_u = pd.concat([iso,u], axis=1)

        # write out the iso_u file for one region
        combined_iso_u_fileName = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Atril_Biosca_Cermoi_iso_u_sca_{sca}\{curRegion}_iso_u.csv'
        #print(combined_iso_u_fileName)
        cur_combined_iso_u.to_csv(combined_iso_u_fileName,index_label='subjID')   
        
        ###  merging with DB INFO  ###
        combined_iso_u_INFO = pd.merge(
        cur_combined_iso_u,
        Atril_Biosca_Cermoi_INFO,
        left_index=True,  #  subjID is the index name of left df
        right_index=True  # subjID is the index name of the right df
        )
        # write out the iso_u file with INFO for one region
        combined_iso_u_INFO_fileName = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Atril_Biosca_Cermoi_iso_u_with_info_sca_{sca}\{curRegion}_iso_u_INFO.csv'
        print(combined_iso_u_INFO_fileName)
        combined_iso_u_INFO.to_csv(combined_iso_u_INFO_fileName,index_label='subjID')


Generating isomaps.
manhattan
manhattan
Generating UMAPs.


C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_iso_u_with_info_sca_1\SC-sylv_right_name06-17-02--84_embeddings_iso_u_INFO.csv
Generating isomaps.
manhattan
manhattan
Generating UMAPs.
C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_iso_u_with_info_sca_1\SFint-SR_right_name08-12-43--252_embeddings_iso_u_INFO.csv
Generating isomaps.
manhattan
manhattan
Generating UMAPs.


C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for paralle

C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_iso_u_with_info_sca_1\SCall_left_name19-33-54--73_embeddings_iso_u_INFO.csv
Generating isomaps.
manhattan
manhattan
Generating UMAPs.
C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_iso_u_with_info_sca_1\OCCIPITAL_left_name07-34-40--229_embeddings_iso_u_INFO.csv
Generating isomaps.
manhattan
manhattan
Generating UMAPs.


...

C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_iso_u_with_info_sca_7\SCall-SsP-SintraCing_left_name10-52-10--210_embeddings_iso_u_INFO.csv
Generating isomaps.
manhattan
manhattan
Generating UMAPs.


C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_iso_u_with_info_sca_7\FPO-SCu-ScCal_left_name07-13-21--118_embeddings_iso_u_INFO.csv
Generating isomaps.
manhattan
manhattan
Generating UMAPs.


C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_iso_u_with_info_sca_7\STi-SOTlat_left_name06-17-04--76_embeddings_iso_u_INFO.csv


In [315]:
######### combining left, right and DB info to write complete composed files ##########

# read in all_DB_info csv
infoBaselineName = rf'{curRoot}:\B_projWIP\proj_{curProject}\SCA_INFO\processed_INFO\ATRIL_BIOSCA_CERMOI_time1_only.csv'
Atril_Biosca_Cermoi_INFO = pd.read_csv(infoBaselineName,index_col=0,header=0)

for curRegion in left:
    region_without_leftRight = strip_after_side(curRegion)
    region_left = curRegion
    matches = [s for s in right if region_without_leftRight in s]
    region_right = matches[0]

    #print(region_left)
    #print(region_right)

    # read in left and right files for a given region
    # iso left
    isoName_left = 'iso_'+region_left+'_dim_'+str(numDim_iso)+'_neig_'+str(numNeig_list_iso)+'.csv'
    Atril_isoFileName_left = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Atril_iso_u\{isoName_left}'
    Atril_iso_left = pd.read_csv(Atril_isoFileName_left,index_col=0,header=0)
    Biosca_isoFileName_left = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Biosca_iso_u\{isoName_left}'
    Biosca_iso_left = pd.read_csv(Biosca_isoFileName_left,index_col=0,header=0)
    Cermoi_isoFileName_left = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Cermoi_iso_u\{isoName_left}'
    Cermoi_iso_left = pd.read_csv(Cermoi_isoFileName_left,index_col=0,header=0)
    # iso right
    isoName_right = 'iso_'+curRegion_right+'_dim_'+str(numDim_iso)+'_neig_'+str(numNeig_list_iso)+'.csv'
    Atril_isoFileName_right = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Atril_iso_u\{isoName_right}'
    Atril_iso_right = pd.read_csv(Atril_isoFileName_right,index_col=0,header=0)
    Biosca_isoFileName_right = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Biosca_iso_u\{isoName_right}'
    Biosca_iso_right = pd.read_csv(Biosca_isoFileName_right,index_col=0,header=0)
    Cermoi_isoFileName_right = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Cermoi_iso_u\{isoName_right}'
    Cermoi_iso_right = pd.read_csv(Cermoi_isoFileName_right,index_col=0,header=0)
    # u left
    uName_left = 'u_'+region_left+'_dim_'+str(numDim_u)+'_neig_'+str(numNeig_list_u)+'.csv'
    Atril_uFileName_left = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Atril_iso_u\{uName_left}'
    Atril_u_left = pd.read_csv(Atril_uFileName_left,index_col=0,header=0)    
    Biosca_uFileName_left = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Biosca_iso_u\{uName_left}'
    Biosca_u_left = pd.read_csv(Biosca_uFileName_left,index_col=0,header=0)    
    Cermoi_uFileName_left = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Cermoi_iso_u\{uName_left}'
    Cermoi_u_left = pd.read_csv(Cermoi_uFileName_left,index_col=0,header=0)    
    # u right
    uName_right = 'u_'+region_right+'_dim_'+str(numDim_u)+'_neig_'+str(numNeig_list_u)+'.csv'
    Atril_uFileName_right = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Atril_iso_u\{uName_right}'
    Atril_u_right = pd.read_csv(Atril_uFileName_right,index_col=0,header=0)    
    Biosca_uFileName_right = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Biosca_iso_u\{uName_right}'
    Biosca_u_right = pd.read_csv(Biosca_uFileName_right,index_col=0,header=0)    
    Cermoi_uFileName_right = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Cermoi_iso_u\{uName_right}'
    Cermoi_u_right = pd.read_csv(Cermoi_uFileName_right,index_col=0,header=0)    
    
    # merge iso and u of left and right separately
    Atril_iso_left = Atril_iso_left.add_prefix('iso_')
    Atril_u_left = Atril_u_left.add_prefix('u_')
    Atril_left = pd.concat([Atril_iso_left,Atril_u_left], axis=1)
    Atril_iso_right = Atril_iso_right.add_prefix('iso_')
    Atril_u_right = Atril_u_right.add_prefix('u_')
    Atril_right = pd.concat([Atril_iso_right,Atril_u_right], axis=1)

    Biosca_iso_left = Biosca_iso_left.add_prefix('iso_')
    Biosca_u_left = Biosca_u_left.add_prefix('u_')
    Biosca_left = pd.concat([Biosca_iso_left,Biosca_u_left], axis=1)
    Biosca_iso_right = Biosca_iso_right.add_prefix('iso_')
    Biosca_u_right = Biosca_u_right.add_prefix('u_')
    Biosca_right = pd.concat([Biosca_iso_right,Biosca_u_right], axis=1)

    Cermoi_iso_left = Cermoi_iso_left.add_prefix('iso_')
    Cermoi_u_left = Cermoi_u_left.add_prefix('u_')
    Cermoi_left = pd.concat([Cermoi_iso_left,Cermoi_u_left], axis=1)
    Cermoi_iso_right = Cermoi_iso_right.add_prefix('iso_')
    Cermoi_u_right = Cermoi_u_right.add_prefix('u_')
    Cermoi_right = pd.concat([Cermoi_iso_right,Cermoi_u_right], axis=1)   
    
    # add subjName column, merge left and right 
    Atril_left['subjName'] = 'L' + Atril_left.index.astype(str)  # add a new column by modifying current index
    Atril_left['Hemisphere'] = 'Left'
    Atril_right['subjName'] = 'flip-R' + Atril_right.index.astype(str)
    Atril_left['Hemisphere'] = 'Right'
    Atril_left = Atril_left.reset_index().set_index('subjName')  # make the new column the index while keeping the old index as a column
    Atril_right = Atril_left.reset_index().set_index('subjName')
    Atril = pd.concat([Atril_left, Atril_right], axis=0)

    Biosca_left['subjName'] = 'L' + Biosca_left.index.astype(str)  # add a new column by modifying current index
    Biosca_left['Hemisphere'] = 'Left'
    Biosca_right['subjName'] = 'flip-R' + Biosca_right.index.astype(str)
    Biosca_right['Hemisphere'] = 'Right'
    Biosca_left = Biosca_left.reset_index().set_index('subjName')  # make the new column the index while keeping the old index as a column
    Biosca_right = Biosca_left.reset_index().set_index('subjName')
    Biosca = pd.concat([Biosca_left, Biosca_right], axis=0)

    Cermoi_left['subjName'] = 'L' + Cermoi_left.index.astype(str)  # add a new column by modifying current index
    Cermoi_left['Hemisphere'] = 'Left'
    Cermoi_right['subjName'] = 'flip-R' + Cermoi_right.index.astype(str)
    Cermoi_right['Hemisphere'] = 'Right'
    Cermoi_left = Cermoi_left.reset_index().set_index('subjName')  # make the new column the index while keeping the old index as a column
    Cermoi_right = Cermoi_left.reset_index().set_index('subjName')
    Cermoi = pd.concat([Cermoi_left, Cermoi_right], axis=0)
    
    # merge Atril, Biosca and Cermoi
    Atril_Biosca_Cermoi = pd.concat([Atril, Biosca, Cermoi], axis=0)
    
    # combine with all_DB_info
    Atril_Biosca_Cermoi_combined = pd.merge(
    Atril_Biosca_Cermoi,
    Atril_Biosca_Cermoi_INFO,
    left_on='subjID',        # Column from Atril_Biosca_Cermoi to use for joining
    right_index=True,        # Use the index of Atril_Biosca_Cermoi for joining
    how='left'               # Keep all Atril_Biosca_Cermoi subjects
    )
    
    # write out the combined file
    outName = region_without_leftRight + '.csv'
    outFileName = rf'{curRoot}:\B_projWIP\proj_{curProject}\Champollion\Atril_Biosca_Cermoi_combined\{outName}'
    print(outFileName)
    Atril_Biosca_Cermoi_combined.to_csv(outFileName,index_label='subjName')


C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_combined\SCall.csv
C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_combined\OCCIPITAL.csv
C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_combined\SPeC.csv
C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_combined\SOr-SOlf.csv
C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_combined\SFmedian-SFpoltr-SFsup.csv
C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_combined\ScCal-SLi.csv
C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_combined\SC-sylv.csv
C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_combined\SFint-FCMant.csv
C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_combined\SFint-SR.csv
C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_combined\STsbr.csv
C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_combined\FIP-FIPPoCinf.csv
C:\B_projWIP\proj_ataxia\Champollion\Atril_Biosca_Cermoi_combined\Lobule_parietal_sup.csv
C:\B_projWIP\proj_ataxia\Ch